# Imports de las librerías

In [32]:
# Imports de las librerías
import sqlite3
from bs4 import BeautifulSoup
import requests
import time
import re

# Se crean las tablas
1) categories
2) authors
3) books
4) book_author


In [33]:
# FK activadas manualmente
cursor.execute("PRAGMA foreign_keys = ON")

# Tabla de categories
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        name    TEXT UNIQUE   NOT NULL
    )
""")

# Table de authors
cursor.execute("""
    CREATE TABLE IF NOT EXISTS authors (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        name              TEXT    NOT NULL,
        birth_year        INTEGER,
        country           TEXT,
        external_api_id   TEXT    UNIQUE,
        total_known_works INTEGER,
        api_source        TEXT,
        created_at        DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

# Tabla de books
cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        title       TEXT    NOT NULL,
        price       REAL    NOT NULL CHECK (price >= 0),
        rating      INTEGER NOT NULL CHECK (rating >= 1 AND rating <= 5),
        category_id INTEGER NOT NULL,
        url         TEXT,
        created_at  DATETIME DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (category_id) REFERENCES categories(id)
    )
""")

# Tabla book_author (tabla puente N:M) 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS book_author (
        book_id   INTEGER NOT NULL,
        author_id INTEGER NOT NULL,
        PRIMARY KEY (book_id, author_id),
        FOREIGN KEY (book_id)   REFERENCES books(id)   ON DELETE CASCADE,
        FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE
    )
""")

connection.commit()
connection.close()
print("Base de datos creada: pengu_books.db")

Base de datos creada: pengu_books.db


# Conexión a la BD (SQLite como DBMS)

In [34]:
# Conexión a la BD
connection = sqlite3.connect("pengu_books.db")
cursor = connection.cursor()
print("Se realiza la conexión a la BD pengu_books")

Se realiza la conexión a la BD pengu_books


# Checking de la página "Books to Scrape"

In [35]:
# Checking rápido si responde la página de Books to Scrape

try:
    # Indicamos la URL 
    url_books_to_scrape = "https://books.toscrape.com/"

    # Se guarda la respuesta del método GET en la variable response
    url_response = requests.get(url_books_to_scrape)

    # Verificación del estado de conexión
    if url_response.status_code == 200:
        print("Conexión exitosa. La página Books to Scrape responde")

except (Exception,KeyboardInterrupt) as error:
    print(f"Hubo un error {error}")

Conexión exitosa. La página Books to Scrape responde


# Variable Global


In [36]:
URL = "https://books.toscrape.com/"

# Extracción de las categorías

In [37]:
# Se inicia extrayendo inicialmente las categorías de los libros
response = requests.get(URL)
sopa = BeautifulSoup(response.text, "html.parser")

# Capturar la lista del panel lateral
lista_categorias = sopa.find("ul", class_="nav nav-list").find("li").find_all("li")

categorias_data = []
for categoria in lista_categorias:          
    nombre = categoria.text.strip()
    link = URL + categoria.a["href"]
    categorias_data.append((nombre, link))

cursor.executemany("INSERT OR IGNORE INTO categories (name) VALUES (?)", [(c[0],) for c in categorias_data])
connection.commit()

# Caché de mapeo id ↔ nombre
cursor.execute("SELECT id, name FROM categories")
categorias_db = {row[1]: row[0] for row in cursor.fetchall()}

print(f"{len(categorias_db)} categorías guardadas")


50 categorías guardadas


# Extracción de libros

In [39]:
rating = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
libros = []

def scrappear_categoria(nombre_categoria, url_categoria, id_categoria):
    libros_scrappeados = []
    url_actual = url_categoria

    while url_actual:
        response = requests.get(url_actual)
        sopa_categoria = BeautifulSoup(response.text, "html.parser")
        articulos = sopa_categoria.find_all("article", class_="product_pod")

        for articulo in articulos:                                              
            titulo = articulo.h3.a["title"].strip()
            precio = articulo.find("p", class_="price_color").text
            precio = float(precio.replace("£", "").replace("Â", "").strip())
            class_rating = articulo.find("p", class_="star-rating")["class"][1]
            estrellas = rating.get(class_rating, 0)

            link_relativo = articulo.h3.a["href"].replace("../../../", "")      
            url_libro = f"https://books.toscrape.com/catalogue/{link_relativo}"

            libros_scrappeados.append((titulo, precio, estrellas, id_categoria, url_libro))  

        boton_siguiente = sopa_categoria.find("li", class_="next")              
        if boton_siguiente:
            url_actual = url_actual.rsplit("/", 1)[0] + "/" + boton_siguiente.a["href"]
        else:
            url_actual = None

    return libros_scrappeados

print("Extrayendo las categorías")
tiempo_inicial = time.time()

for nombre, url in categorias_data:
    id_categoria = categorias_db[nombre]
    libros_de_cada_categoria = scrappear_categoria(nombre, url, id_categoria)
    libros.extend(libros_de_cada_categoria)

cursor.executemany("""
    INSERT OR IGNORE INTO books (title, price, rating, category_id, url)
    VALUES (?, ?, ?, ?, ?)
""", libros)
connection.commit()

# ❌ Borrar esto
# cursor.executemany("""
#     INSERT INTO books (title, price, rating, category_id, url)
#     VALUES (?, ?, ?, ?, ?)
# """, libros)

tiempo_final = time.time()                                                      
print(f"{len(libros)} libros extraídos")
print(f"El tiempo de inserción fue de {round(tiempo_final - tiempo_inicial)} segundos")

Extrayendo las categorías
1000 libros extraídos
El tiempo de inserción fue de 60 segundos
